# Atrasos de entrega e avaliação dos clientes

**Projeto:** Kaggle + Artigo Técnico  
**Fonte:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

## Pergunta

O que aparece associado a atrasos nas entregas e à avaliação dos clientes?

Vamos comparar a data real com a data estimada para pedidos entregues e observar como as notas de avaliação variam entre entregas atrasadas e entregas no prazo ou adiantadas.

> O conjunto da Olist descreve comércio eletrônico brasileiro, não pedidos de restaurantes. As conclusões não representam o iFood. A análise é descritiva e não demonstra causalidade.

## Como adicionar os dados no Kaggle

No Kaggle, use **Add Input**, pesquise por **Brazilian E-Commerce Public Dataset by Olist** e adicione o conjunto. Localmente, coloque os CSVs em `data/raw/`. Os arquivos brutos não devem ser enviados ao repositório. A página do Kaggle informa licença CC BY-NC-SA 4.0; consulte os termos e dê crédito à fonte.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 1. Encontrar e carregar os arquivos

O código procura primeiro a pasta de entrada do Kaggle e depois `data/raw/`, para permitir execução no Kaggle ou localmente.

In [ ]:
DATA_FILES = {
    "orders": "olist_orders_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
}

SEARCH_ROOTS = [Path("/kaggle/input"), Path("data/raw")]


def find_dataset_file(filename: str) -> Path:
    for root in SEARCH_ROOTS:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        f"Não encontrei {filename}. No Kaggle, adicione o dataset Olist como input. "
        "Localmente, coloque os CSVs em data/raw/."
    )


file_paths = {name: find_dataset_file(filename) for name, filename in DATA_FILES.items()}
file_paths

In [ ]:
orders = pd.read_csv(file_paths["orders"])
reviews = pd.read_csv(file_paths["reviews"])
customers = pd.read_csv(file_paths["customers"])

print(f"Pedidos: {orders.shape[0]:,} linhas, {orders.shape[1]} colunas")
print(f"Avaliações: {reviews.shape[0]:,} linhas, {reviews.shape[1]} colunas")
print(f"Clientes: {customers.shape[0]:,} linhas, {customers.shape[1]} colunas")

orders.head()

## 2. Verificações iniciais de qualidade

Antes de calcular resultados, verificamos chaves repetidas, datas ausentes e o período coberto. Os CSVs podem conter pedidos ainda não entregues ou sem todos os registros de data; esses casos serão tratados explicitamente.

In [ ]:
quality_checks = pd.DataFrame(
    {
        "linhas": [len(orders), len(reviews), len(customers)],
        "order_id_duplicado": [
            orders["order_id"].duplicated().sum(),
            reviews["order_id"].duplicated().sum(),
            pd.NA,
        ],
    },
    index=["orders", "reviews", "customers"],
)
quality_checks

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"], errors="coerce"
)
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"], errors="coerce"
)

date_coverage = orders["order_purchase_timestamp"].agg(["min", "max"])
missing_dates = orders[
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
].isna().sum()

display(date_coverage.to_frame("data de compra"))
display(missing_dates.to_frame("valores ausentes"))

## 3. Preparar uma linha por pedido entregue

As avaliações são agregadas por pedido antes da junção. Assim, caso um pedido tenha mais de um registro de avaliação, a tabela final continua com uma linha por pedido.

In [ ]:
review_by_order = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "nunique"),
    )
)

customer_state = customers[["customer_id", "customer_state"]].drop_duplicates(
    "customer_id"
)

analysis = (
    orders.merge(customer_state, on="customer_id", how="left", validate="many_to_one")
    .merge(review_by_order, on="order_id", how="left", validate="one_to_one")
)

delivered = analysis.loc[
    analysis["order_status"].eq("delivered")
    & analysis["order_delivered_customer_date"].notna()
    & analysis["order_estimated_delivery_date"].notna()
].copy()

delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"]
    - delivered["order_purchase_timestamp"]
).dt.total_seconds() / 86_400

delivered["delay_days"] = (
    delivered["order_delivered_customer_date"]
    - delivered["order_estimated_delivery_date"]
).dt.total_seconds() / 86_400

delivered["delivery_timing"] = delivered["delay_days"].gt(0).map(
    {True: "Atrasado", False: "No prazo ou adiantado"}
)

print(f"Pedidos analisados: {len(delivered):,}")
print(f"Pedidos com nota: {delivered['review_score'].notna().sum():,}")
delivered[["order_id", "customer_state", "delay_days", "delivery_timing", "review_score"]].head()

## 4. Resumo dos atrasos e das avaliações

Atraso é definido como entrega real posterior à data estimada. A taxa abaixo usa somente pedidos entregues que têm as duas datas necessárias. O notebook calcula os resultados a partir dos dados carregados; nenhum número é fixado previamente.

In [ ]:
late_mask = delivered["delay_days"].gt(0)
late_reviews = delivered.loc[late_mask, "review_score"].dropna()
on_time_reviews = delivered.loc[~late_mask, "review_score"].dropna()

summary = pd.Series(
    {
        "pedidos_entregues_com_datas": len(delivered),
        "pedidos_atrasados": int(late_mask.sum()),
        "percentual_atrasado": late_mask.mean() * 100 if len(delivered) else float("nan"),
        "mediana_atraso_dias_todos": delivered["delay_days"].median(),
        "mediana_dias_de_atraso_apenas_atrasados": (
            delivered.loc[late_mask, "delay_days"].median()
            if late_mask.any()
            else float("nan")
        ),
        "nota_media_atrasados": late_reviews.mean(),
        "nota_media_no_prazo_ou_adiantados": on_time_reviews.mean(),
        "cobertura_de_nota_percentual": (
            delivered["review_score"].notna().mean() * 100
            if len(delivered)
            else float("nan")
        ),
    },
    name="valor",
)
summary.to_frame()

## 5. Visualizar a distribuição dos atrasos

O gráfico limita o eixo aos percentis 1 e 99 somente para facilitar a leitura. As observações não são removidas do dataframe nem dos resumos anteriores.

In [ ]:
lower, upper = delivered["delay_days"].quantile([0.01, 0.99])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(delivered["delay_days"].dropna(), bins=50, color="#386cb0", edgecolor="white")
ax.set_xlim(lower, upper)
ax.axvline(0, color="#d95f02", linestyle="--", label="Data estimada")
ax.set_title("Diferença entre entrega real e data estimada")
ax.set_xlabel("Dias (positivo = entrega após a estimativa)")
ax.set_ylabel("Número de pedidos")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Comparar notas por situação de entrega

A comparação é descritiva. Diferenças observadas não provam que o atraso causou uma nota menor; outros fatores podem influenciar a experiência do cliente.

In [ ]:
review_summary = (
    delivered.dropna(subset=["review_score"])
    .groupby("delivery_timing")["review_score"]
    .agg(["count", "mean", "median"])
    .reindex(["No prazo ou adiantado", "Atrasado"])
)
review_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
delivered.boxplot(
    column="review_score",
    by="delivery_timing",
    ax=ax,
    grid=False,
    showfliers=False,
)
ax.set_title("Notas por situação de entrega")
fig.suptitle("")
ax.set_xlabel("")
ax.set_ylabel("Nota da avaliação (1 a 5)")
plt.tight_layout()
plt.show()

## 7. Comparar estados com mais pedidos

Mostramos os dez estados com maior volume de pedidos elegíveis. A tabela inclui a quantidade analisada para que taxas baseadas em grupos pequenos não sejam interpretadas sem contexto.

In [ ]:
state_summary = (
    delivered.groupby("customer_state")
    .agg(
        orders=("order_id", "nunique"),
        late_rate=("delay_days", lambda values: values.gt(0).mean() * 100),
        mean_review=("review_score", "mean"),
    )
    .sort_values("orders", ascending=False)
    .head(10)
)
state_summary

## 8. Anotar resultados para o artigo

Use os valores de `summary` e `review_summary` para preencher `docs/artigo-tecnico.md`. Registre também o tamanho da amostra e os padrões que observou. Explique valores ausentes, recortes e limitações. Não descreva associações como causas.

In [ ]:
print("Resumo calculado para a execução atual:")
display(summary.to_frame())
print("Comparação das notas:")
display(review_summary)